In [0]:
%pip install openpyxl

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd
import os
import re

# 1. Define your paths (Update these to match your Databricks environment)
excel_path = "/Volumes/workspace/default/gst-fraud&anomaly_detection/GST_Anomaly_Dataset_NexuSolve.xlsx"
output_directory = "/Volumes/workspace/default/gst-fraud&anomaly_detection/data/"

# Create the target directory if it doesn't exist
os.makedirs(output_directory, exist_ok=True)

# 2. Load the Excel File structure
excel_file = pd.ExcelFile(excel_path)

print(f"Found sheets: {excel_file.sheet_names}\n")

# 3. Loop through each sheet and convert to CSV
for sheet_name in excel_file.sheet_names:
    print(f"Processing: {sheet_name}...")
    
    # Read the specific sheet into a pandas DataFrame
    df = pd.read_excel(excel_file, sheet_name=sheet_name)
    
    # 4. Clean up the sheet name for file system compatibility
    # This removes emojis (like 📋, 📦) and strips extra spaces
    clean_name = re.sub(r'[^\w\s-]', '', sheet_name).strip().replace(' ', '_')
    
    # Handle cases where stripping emojis leaves the name blank
    if not clean_name:
        clean_name = "sheet"
        
    csv_file_name = f"{clean_name}.csv"
    csv_path = os.path.join(output_directory, csv_file_name)
    
    # 5. Save out to CSV format
    df.to_csv(csv_path, index=False, encoding='utf-8')
    print(f"→ Successfully saved as: {csv_path}")

print("\n🎉 Conversion completed successfully!")

Found sheets: ['📋 README', '📦 PO_Records', '🏢 Vendor_Master', '📊 HSN_Rate_Schedule', '🚫 CBIC_Blacklist', '🏷️ Ground_Truth']

Processing: 📋 README...
→ Successfully saved as: /Volumes/workspace/default/gst-fraud&anomaly_detection/data/README.csv
Processing: 📦 PO_Records...
→ Successfully saved as: /Volumes/workspace/default/gst-fraud&anomaly_detection/data/PO_Records.csv
Processing: 🏢 Vendor_Master...
→ Successfully saved as: /Volumes/workspace/default/gst-fraud&anomaly_detection/data/Vendor_Master.csv
Processing: 📊 HSN_Rate_Schedule...
→ Successfully saved as: /Volumes/workspace/default/gst-fraud&anomaly_detection/data/HSN_Rate_Schedule.csv
Processing: 🚫 CBIC_Blacklist...
→ Successfully saved as: /Volumes/workspace/default/gst-fraud&anomaly_detection/data/CBIC_Blacklist.csv
Processing: 🏷️ Ground_Truth...
→ Successfully saved as: /Volumes/workspace/default/gst-fraud&anomaly_detection/data/Ground_Truth.csv

🎉 Conversion completed successfully!


In [0]:
po_df = spark.read.option("header", True).csv("/Volumes/workspace/default/gst-fraud&anomaly_detection/data/PO_Records.csv")

vendor_df = spark.read.option("header", True).csv("/Volumes/workspace/default/gst-fraud&anomaly_detection/data/Vendor_Master.csv")

hsn_df = spark.read.option("header", True).csv("/Volumes/workspace/default/gst-fraud&anomaly_detection/data/HSN_Rate_Schedule.csv")

blacklist_df = spark.read.option("header", True).csv("/Volumes/workspace/default/gst-fraud&anomaly_detection/data/CBIC_Blacklist.csv")

ground_truth_df = spark.read.option("header", True).csv("/Volumes/workspace/default/gst-fraud&anomaly_detection/data/Ground_Truth.csv")


In [0]:
print("===== Row Count =====")
print("PO Records :", po_df.count())
print("Vendor Master :", vendor_df.count())
print("HSN Rate :", hsn_df.count())
print("Blacklist :", blacklist_df.count())
print("Ground Truth :", ground_truth_df.count())



display(po_df.limit(5))
display(vendor_df.limit(5))
display(hsn_df.limit(5))
display(blacklist_df.limit(5))
display(ground_truth_df.limit(5))





===== Row Count =====
PO Records : 1000
Vendor Master : 80
HSN Rate : 20
Blacklist : 20
Ground Truth : 1001


PO ID,PO Date,Inv Date,Invoice No.,Vendor ID,Vendor GSTIN,Trade Name,Billing Name,Billing State,Buyer GSTIN,Buyer State,HSN Code,Product,Qty,Unit,Base Amt (₹),CGST%,SGST%,IGST%,CGST Amt,SGST Amt,IGST Amt,Cess,Total Inv (₹),EWB No.,EWB Date,Place of Supply,Delivery State,GSTIN Reg Date,GSTIN Status,Cancellation Date,Composition?,Filing 6Q,GSTR2B ITC (₹),ITC Claimed (₹),6M Avg (₹),Dir PAN 1,Anomaly Codes,Severity,Analyst Notes
PO-2024-00001,05/01/2023,06/01/2023,INV-VND0035-0123-436,VND0035,05QQQQQ6439D1Z1,Premier Systems Enterprises,Premier Systems Enterprises,Uttarakhand,27IIIII3249D1ZL,Maharashtra,72044900,Steel scrap,184,MTR,4267393.69,0.0,0.0,18,0.0,0.0,768130.86,0,5035524.55,EWB812272035,06/01/2023,Maharashtra,Maharashtra,15/11/2021,active,null,No,3/6,768130.86,768130.86,3855939.58,GGGGG2139B,ITC-001,Critical,Vendor has not filed GSTR-1/3B in 3+ of last 6 quarters — ITC at risk under Sec 37A
PO-2024-00002,05/12/2011,08/12/2011,INV-VND0032-0224-268,VND0032,21WWWWW8941D1ZY,Supreme Electronics Private Limited,Supreme Electronics Private Limited,Odisha,27IIIII3249D1ZL,Maharashtra,85414010,Solar panels,161,RLS,3862225.77,0.0,0.0,5,0.0,0.0,193111.29,0,4055337.06,EWB217008688,03/02/2024,Maharashtra,Maharashtra,24/08/2012,active,null,No,6/6,193111.29,193111.29,1202813.59,UUUUU3803R,GST-004,Critical,Invoice date 08/12/2011 is before GSTIN registration 24/08/2012
PO-2024-00003,01/08/2023,02/08/2023,INV-VND0033-0823-111,VND0033,08OOOOO3369M1ZR,Indian Energy Works,Indian Energy Works,Rajasthan,27IIIII3249D1ZL,Maharashtra,30049099,Pharmaceuticals,354,SET,3222857.55,0.0,0.0,12,0.0,0.0,386742.91,0,3609600.46,null,null,Maharashtra,Maharashtra,10/04/2016,active,null,No,6/6,251940.44,386742.91,217208.82,BBBBB8527R,EWB-001,High,"No e-Way Bill found for goods PO worth ₹3,222,858 (threshold: ₹50,000)"
PO-2024-00004,02/03/2023,02/03/2023,INV-VND0031-0324-575,VND0031,24BBBBB8451C1Z2,Capital Components Traders,Capital Components Traders,Gujarat,27IIIII3249D1ZL,Maharashtra,76061110,Aluminium sheets,395,LTR,117891.26,0.0,0.0,18,0.0,0.0,687165.26,0,139111.69,null,18/03/2024,Maharashtra,Maharashtra,20/01/2019,active,null,No,4/6,687165.26,687165.26,1667191.47,DDDDD9935C,FRD-003,High,"Split invoice: ₹117,891 + ₹80,126 = ₹198,017 on same day from same vendor — threshold avoidance"
PO-2024-00005,23/08/2023,26/08/2023,INV-VND0058-0823-423,VND0058,22GGGGG4711E1ZG,National Energy Corporation,National Energy Corporation,Chhattisgarh,27IIIII3249D1ZL,Maharashtra,85423100,Semiconductors,5,MT,4883710.62,0.0,0.0,28,0.0,0.0,1367438.97,0,6251149.59,EWB849734394,24/08/2023,Maharashtra,Maharashtra,23/06/2015,active,null,No,4/6,879067.91,1367438.97,3453071.29,UUUUU9928N,TAX-001,Critical,HSN 85423100 should attract 18% GST but invoice charges 28%


Vendor ID,GSTIN,Trade Name,Legal Name,Billing State,State Code,Reg Date,Status,Cancellation Date,Composition?,Q1,Q2,Q3,Q4,Q5,Q6,Director PAN 1,Director PAN 2,Monthly Avg PO (₹)
VND0001,23AAAAA6310P1ZY,Supreme Chemicals Manufacturing,Supreme Chemicals Manufacturing,Madhya Pradesh,23,08/06/2012,cancelled,04/08/2023,No,✗,✓,✓,✓,✓,✓,UUUUU6573D,UUUUU8517E,812458.18
VND0002,20GGGGG9835Y1ZW,Infra Chemicals Trading Co,Infra Chemicals Trading Co Reg.,Jharkhand,20,05/07/2016,cancelled,11/09/2022,No,✓,✓,✓,✓,✓,✓,XXXXX5506H,SSSSS7912B,4500963.48
VND0003,23HHHHH5562V1ZX,Tata Plastics Private Limited,Tata Plastics Private Limited,Madhya Pradesh,23,20/11/2022,cancelled,21/02/2023,No,✓,✗,✓,✓,✓,✓,MMMMM6930H,VVVVV7916T,3809448.35
VND0004,11ZZZZZ2790H1ZB,General Plastics Pvt Ltd,General Plastics Pvt Ltd,Sikkim,11,18/04/2016,cancelled,30/03/2023,No,✓,✓,✓,✓,✓,✓,XXXXX8019S,OOOOO6977F,5318123.26
VND0005,02FFFFF7209A1ZQ,Lakshmi Systems Ltd,Lakshmi Systems Ltd,Himachal Pradesh,2,03/10/2020,cancelled,24/07/2023,No,✓,✓,✓,✓,✓,✓,NNNNN6574I,QQQQQ4258E,2268579.23


HSN Code,Product Description,GST Rate %,Cess?,Effective From
72101200,Steel coils,18,No,01/07/2017
72044900,Steel scrap,18,No,01/07/2017
39201020,Plastic film,18,No,01/07/2017
39269099,Plastic parts,18,No,01/07/2017
85414010,Solar panels,5,No,01/07/2017


Blacklisted GSTIN,Blacklist Date,Reason,Case Reference
23AAAAA6310P1ZY,06/08/2023,Shell company,CBIC-GST-6409/2023
20GGGGG9835Y1ZW,05/10/2021,Shell company,CBIC-GST-9591/2023
23HHHHH5562V1ZX,29/08/2023,Fake ITC fraud,CBIC-GST-6205/2022
11ZZZZZ2790H1ZB,03/01/2022,Shell company,CBIC-GST-8708/2021
02FFFFF7209A1ZQ,12/01/2021,Non-existent business,CBIC-GST-4074/2021


⚠ ANSWER KEY — Split this sheet off BEFORE training your ML model. Do not use this as an input feature.,Unnamed: 1,Unnamed: 2,Unnamed: 3
PO ID,Anomaly Code(s),Severity,is_anomalous (0/1)
PO-2024-00001,ITC-001,Critical,1
PO-2024-00002,GST-004,Critical,1
PO-2024-00003,EWB-001,High,1
PO-2024-00004,FRD-003,High,1


In [0]:
for name, df in {
    "PO": po_df,
    "Vendor": vendor_df,
    "HSN": hsn_df,
    "Blacklist": blacklist_df,
    "GroundTruth": ground_truth_df
}.items():
    print(f"\n===== {name} =====")
    print("Columns:")
    print(df.columns)
    print("Row Count:", df.count())
    df.show(3, truncate=False)


===== PO =====
Columns:
['PO ID', 'PO Date', 'Inv Date', 'Invoice No.', 'Vendor ID', 'Vendor GSTIN', 'Trade Name', 'Billing Name', 'Billing State', 'Buyer GSTIN', 'Buyer State', 'HSN Code', 'Product', 'Qty', 'Unit', 'Base Amt (₹)', 'CGST%', 'SGST%', 'IGST%', 'CGST Amt', 'SGST Amt', 'IGST Amt', 'Cess', 'Total Inv (₹)', 'EWB No.', 'EWB Date', 'Place of Supply', 'Delivery State', 'GSTIN Reg Date', 'GSTIN Status', 'Cancellation Date', 'Composition?', 'Filing 6Q', 'GSTR2B ITC (₹)', 'ITC Claimed (₹)', '6M Avg (₹)', 'Dir PAN 1', 'Anomaly Codes', 'Severity', 'Analyst Notes']
Row Count: 1000
+-------------+----------+----------+--------------------+---------+---------------+-----------------------------------+-----------------------------------+-------------+---------------+-----------+--------+---------------+---+----+------------+-----+-----+-----+--------+--------+---------+----+-------------+------------+----------+---------------+--------------+--------------+------------+----------------

##Ground Truth 

In [0]:
ground_truth_raw = spark.read.option("header", True).csv("/Volumes/workspace/default/gst-fraud&anomaly_detection/data/Ground_Truth.csv")

ground_truth_pd = ground_truth_raw.toPandas()

ground_truth_pd.columns = [
    "warning",
    "col2",
    "col3",
    "col4"
]

ground_truth_pd = ground_truth_pd.iloc[1:]

ground_truth_pd.columns = [
    "po_id",
    "anomaly_code",
    "severity",
    "is_anomalous"
]

ground_truth_df = spark.createDataFrame(ground_truth_pd)

display(ground_truth_df.limit(5))

po_id,anomaly_code,severity,is_anomalous
PO-2024-00001,ITC-001,Critical,1
PO-2024-00002,GST-004,Critical,1
PO-2024-00003,EWB-001,High,1
PO-2024-00004,FRD-003,High,1
PO-2024-00005,TAX-001,Critical,1


In [0]:
ground_truth_df = ground_truth_df.withColumn(
    "anomaly_code",
    when(
        col("anomaly_code") == "FRD-006",
        "FRD-002"
    ).otherwise(col("anomaly_code"))
)

In [0]:
ground_truth_df.count()
ground_truth_df.show(5)

+-------------+------------+--------+------------+
|        po_id|anomaly_code|severity|is_anomalous|
+-------------+------------+--------+------------+
|PO-2024-00001|     ITC-001|Critical|           1|
|PO-2024-00002|     GST-004|Critical|           1|
|PO-2024-00003|     EWB-001|    High|           1|
|PO-2024-00004|     FRD-003|    High|           1|
|PO-2024-00005|     TAX-001|Critical|           1|
+-------------+------------+--------+------------+
only showing top 5 rows


## Vendor CSV

In [0]:
from pyspark.sql import functions as F

vendors_df = vendor_df.select(
    F.col("Vendor ID").alias("vendor_id"),
    F.col("GSTIN").alias("gstin"),
    F.col("Trade Name").alias("trade_name"),
    F.col("Legal Name").alias("legal_name"),
    F.col("Billing State").alias("billing_state"),
    F.col("Reg Date").alias("registration_date"),
    F.col("Status").alias("status"),
    F.col("Composition?").alias("composition_flag"),
    F.col("Q1").alias("filing_history_q1"),
    F.col("Q2").alias("filing_history_q2"),
    F.col("Q3").alias("filing_history_q3"),
    F.col("Q4").alias("filing_history_q4"),
    F.col("Q5").alias("filing_history_q5"),
    F.col("Q6").alias("filing_history_q6"),
    F.col("Director PAN 1").alias("director_pan_1"),
    F.col("Director PAN 2").alias("director_pan_2")
)

display(vendors_df.limit(5))

vendor_id,gstin,trade_name,legal_name,billing_state,registration_date,status,composition_flag,filing_history_q1,filing_history_q2,filing_history_q3,filing_history_q4,filing_history_q5,filing_history_q6,director_pan_1,director_pan_2
VND0001,23AAAAA6310P1ZY,Supreme Chemicals Manufacturing,Supreme Chemicals Manufacturing,Madhya Pradesh,08/06/2012,cancelled,No,✗,✓,✓,✓,✓,✓,UUUUU6573D,UUUUU8517E
VND0002,20GGGGG9835Y1ZW,Infra Chemicals Trading Co,Infra Chemicals Trading Co Reg.,Jharkhand,05/07/2016,cancelled,No,✓,✓,✓,✓,✓,✓,XXXXX5506H,SSSSS7912B
VND0003,23HHHHH5562V1ZX,Tata Plastics Private Limited,Tata Plastics Private Limited,Madhya Pradesh,20/11/2022,cancelled,No,✓,✗,✓,✓,✓,✓,MMMMM6930H,VVVVV7916T
VND0004,11ZZZZZ2790H1ZB,General Plastics Pvt Ltd,General Plastics Pvt Ltd,Sikkim,18/04/2016,cancelled,No,✓,✓,✓,✓,✓,✓,XXXXX8019S,OOOOO6977F
VND0005,02FFFFF7209A1ZQ,Lakshmi Systems Ltd,Lakshmi Systems Ltd,Himachal Pradesh,03/10/2020,cancelled,No,✓,✓,✓,✓,✓,✓,NNNNN6574I,QQQQQ4258E


In [0]:
print("Vendor Count:", vendors_df.count())
print("Columns:", len(vendors_df.columns))

Vendor Count: 80
Columns: 16


## Purchase Order CSV

In [0]:
#PURCHASE ORDER DF

from pyspark.sql import functions as F

purchase_orders_df = po_df.select(
    F.col("PO ID").alias("po_id"),
    F.col("PO Date").alias("po_date"),
    F.col("Vendor ID").alias("vendor_id"),
    F.col("Buyer GSTIN").alias("buyer_gstin"),
    F.col("Buyer State").alias("buyer_state"),
    F.col("HSN Code").alias("hsn_code"),
    F.col("Product").alias("product_desc"),
    F.col("Qty").cast("int").alias("quantity"),
    F.col("Unit").alias("unit"),
    F.col("`Base Amt (₹)`").cast("double").alias("base_amount"),
    F.col("`CGST%`").cast("double").alias("cgst_rate"),
    F.col("`SGST%`").cast("double").alias("sgst_rate"),
    F.col("`IGST%`").cast("double").alias("igst_rate"),
    F.col("CGST Amt").cast("double").alias("cgst_amt"),
    F.col("SGST Amt").cast("double").alias("sgst_amt"),
    F.col("IGST Amt").cast("double").alias("igst_amt"),
    F.col("Cess").cast("double").alias("cess_amt"),
    F.col("`Total Inv (₹)`").cast("double").alias("total_amount"),
    F.col("`EWB No.`").alias("ewb_number"),
    F.col("EWB Date").alias("ewb_generated_date"),
    F.col("Place of Supply").alias("place_of_supply"),
    F.col("Delivery State").alias("delivery_state"),
    F.col("Billing Name").alias("invoice_billing_name"),
    F.col("Trade Name").alias("po_vendor_name")
)

display(purchase_orders_df.limit(5))

po_id,po_date,vendor_id,buyer_gstin,buyer_state,hsn_code,product_desc,quantity,unit,base_amount,cgst_rate,sgst_rate,igst_rate,cgst_amt,sgst_amt,igst_amt,cess_amt,total_amount,ewb_number,ewb_generated_date,place_of_supply,delivery_state,invoice_billing_name,po_vendor_name
PO-2024-00001,05/01/2023,VND0035,27IIIII3249D1ZL,Maharashtra,72044900,Steel scrap,184,MTR,4267393.69,0.0,0.0,18.0,0.0,0.0,768130.86,0.0,5035524.55,EWB812272035,06/01/2023,Maharashtra,Maharashtra,Premier Systems Enterprises,Premier Systems Enterprises
PO-2024-00002,05/12/2011,VND0032,27IIIII3249D1ZL,Maharashtra,85414010,Solar panels,161,RLS,3862225.77,0.0,0.0,5.0,0.0,0.0,193111.29,0.0,4055337.06,EWB217008688,03/02/2024,Maharashtra,Maharashtra,Supreme Electronics Private Limited,Supreme Electronics Private Limited
PO-2024-00003,01/08/2023,VND0033,27IIIII3249D1ZL,Maharashtra,30049099,Pharmaceuticals,354,SET,3222857.55,0.0,0.0,12.0,0.0,0.0,386742.91,0.0,3609600.46,null,null,Maharashtra,Maharashtra,Indian Energy Works,Indian Energy Works
PO-2024-00004,02/03/2023,VND0031,27IIIII3249D1ZL,Maharashtra,76061110,Aluminium sheets,395,LTR,117891.26,0.0,0.0,18.0,0.0,0.0,687165.26,0.0,139111.69,null,18/03/2024,Maharashtra,Maharashtra,Capital Components Traders,Capital Components Traders
PO-2024-00005,23/08/2023,VND0058,27IIIII3249D1ZL,Maharashtra,85423100,Semiconductors,5,MT,4883710.62,0.0,0.0,28.0,0.0,0.0,1367438.97,0.0,6251149.59,EWB849734394,24/08/2023,Maharashtra,Maharashtra,National Energy Corporation,National Energy Corporation


In [0]:
print("PO Count:", purchase_orders_df.count())
print("Columns:", len(purchase_orders_df.columns))

PO Count: 1000
Columns: 24


## vendor_invoices CSV


In [0]:
from pyspark.sql import functions as F

from pyspark.sql import functions as F

vendor_invoices_df = po_df.select(
    F.col("PO ID").alias("po_id"),
    F.col("Inv Date").alias("invoice_date"),
    F.col("`Invoice No.`").alias("invoice_number"),
    F.col("`GSTR2B ITC (₹)`").alias("gstr2b_itc_available"),
    F.col("`ITC Claimed (₹)`").alias("itc_claimed_by_buyer")
)

display(vendor_invoices_df.limit(5))

po_id,invoice_date,invoice_number,gstr2b_itc_available,itc_claimed_by_buyer
PO-2024-00001,06/01/2023,INV-VND0035-0123-436,768130.86,768130.86
PO-2024-00002,08/12/2011,INV-VND0032-0224-268,193111.29,193111.29
PO-2024-00003,02/08/2023,INV-VND0033-0823-111,251940.44,386742.91
PO-2024-00004,02/03/2023,INV-VND0031-0324-575,687165.26,687165.26
PO-2024-00005,26/08/2023,INV-VND0058-0823-423,879067.91,1367438.97


In [0]:
print("Invoice Count:", vendor_invoices_df.count())
vendor_invoices_df.printSchema()


Invoice Count: 1000
root
 |-- po_id: string (nullable = true)
 |-- invoice_date: string (nullable = true)
 |-- invoice_number: string (nullable = true)
 |-- gstr2b_itc_available: string (nullable = true)
 |-- itc_claimed_by_buyer: string (nullable = true)



##hsn_rate_schedule CSV


In [0]:
from pyspark.sql import functions as F

hsn_rate_schedule_df = hsn_df.select(
    F.col("HSN Code").alias("hsn_code"),
    F.col("Product Description").alias("description"),
    F.col("`GST Rate %`").alias("gst_rate"),
    F.col("`Cess?`").alias("cess_applicable"),
    F.col("`Effective From`").alias("effective_from")
)

display(hsn_rate_schedule_df)

hsn_code,description,gst_rate,cess_applicable,effective_from
72101200,Steel coils,18,No,01/07/2017
72044900,Steel scrap,18,No,01/07/2017
39201020,Plastic film,18,No,01/07/2017
39269099,Plastic parts,18,No,01/07/2017
85414010,Solar panels,5,No,01/07/2017
85423100,Semiconductors,18,No,01/07/2017
29051100,Methanol,18,No,01/07/2017
29053100,Ethylene glycol,18,No,01/07/2017
84314900,Machine parts,18,No,01/07/2017
84713020,Laptops,18,No,01/07/2017


In [0]:
print("HSN Count:", hsn_rate_schedule_df.count())

hsn_rate_schedule_df.printSchema()

HSN Count: 20
root
 |-- hsn_code: string (nullable = true)
 |-- description: string (nullable = true)
 |-- gst_rate: string (nullable = true)
 |-- cess_applicable: string (nullable = true)
 |-- effective_from: string (nullable = true)



##cbic blacklist CSV

In [0]:
from pyspark.sql import functions as F

cbic_blacklist_df = blacklist_df.select(
    F.col("Blacklisted GSTIN").alias("gstin"),
    F.col("Blacklist Date").alias("blacklist_date"),
    F.col("Reason").alias("reason"),
    F.col("Case Reference").alias("case_reference")
)

display(cbic_blacklist_df)

gstin,blacklist_date,reason,case_reference
23AAAAA6310P1ZY,06/08/2023,Shell company,CBIC-GST-6409/2023
20GGGGG9835Y1ZW,05/10/2021,Shell company,CBIC-GST-9591/2023
23HHHHH5562V1ZX,29/08/2023,Fake ITC fraud,CBIC-GST-6205/2022
11ZZZZZ2790H1ZB,03/01/2022,Shell company,CBIC-GST-8708/2021
02FFFFF7209A1ZQ,12/01/2021,Non-existent business,CBIC-GST-4074/2021
35PPPPP6128U1ZC,12/07/2021,Forged invoices,CBIC-GST-8513/2022
16KKKKK8559W1ZS,13/07/2021,Fake ITC fraud,CBIC-GST-8679/2023
31QQQQQ2334D1ZK,17/11/2021,Fake ITC fraud,CBIC-GST-6741/2023
28WWWWW2269M1Z8,08/04/2022,Non-existent business,CBIC-GST-1231/2021
09YYYYY1223F1ZP,17/07/2022,Non-existent business,CBIC-GST-7982/2023


In [0]:
#FAKE


from pyspark.sql.functions import rand

cbic_blacklist_df = (
    vendors_df
    .select("gstin")
    .sample(False, 0.05, seed=42)   # ~20 vendors out of 1000
    .withColumn("blacklist_date", F.current_date())
    .withColumn("reason", F.lit("Fake GST Registration"))
    .withColumn("case_reference", F.lit("CBIC-TEST"))
)

In [0]:
print("Blacklist Count:", cbic_blacklist_df.count())

cbic_blacklist_df.printSchema()

Blacklist Count: 4
root
 |-- gstin: string (nullable = true)
 |-- blacklist_date: date (nullable = false)
 |-- reason: string (nullable = false)
 |-- case_reference: string (nullable = false)



##historical po values CSV

In [0]:
from pyspark.sql import functions as F

historical_po_values_df = po_df.select(
    F.col("Vendor ID").alias("vendor_id"),
    F.date_format(
        F.to_date(F.col("PO Date"), "dd/MM/yyyy"),
        "yyyy-MM"
    ).alias("year_month"),
    F.col("`6M Avg (₹)`").alias("total_po_value")
)

historical_po_values_df = historical_po_values_df.groupBy(
    "vendor_id",
    "year_month"
).agg(
    F.avg("total_po_value").alias("total_po_value"),
    F.count("*").alias("po_count")
)

display(historical_po_values_df)

vendor_id,year_month,total_po_value,po_count
VND0033,2023-08,217208.82,1
VND0017,2024-01,1384536.45,3
VND0026,2023-05,636934.37,3
VND0003,2024-02,3809448.35,2
VND0056,2023-08,1053848.43,1
VND0044,2024-03,380813.65,1
VND0013,2023-08,3659844.76,3
VND0037,2023-05,284895.27,2
VND0046,2023-07,1120383.65,2
VND0006,2024-03,921533.91,1


In [0]:
print("Historical Count:", historical_po_values_df.count())

historical_po_values_df.printSchema()

Historical Count: 657
root
 |-- vendor_id: string (nullable = true)
 |-- year_month: string (nullable = true)
 |-- total_po_value: double (nullable = true)
 |-- po_count: long (nullable = false)



##Check

In [0]:
#null Count Check


from pyspark.sql import functions as F

purchase_orders_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in purchase_orders_df.columns
]).show(vertical=True, truncate=False)

-RECORD 0-------------------
 po_id                | 0   
 po_date              | 0   
 vendor_id            | 0   
 buyer_gstin          | 0   
 buyer_state          | 0   
 hsn_code             | 0   
 product_desc         | 0   
 quantity             | 0   
 unit                 | 0   
 base_amount          | 0   
 cgst_rate            | 0   
 sgst_rate            | 0   
 igst_rate            | 0   
 cgst_amt             | 0   
 sgst_amt             | 0   
 igst_amt             | 0   
 cess_amt             | 0   
 total_amount         | 0   
 ewb_number           | 110 
 ewb_generated_date   | 70  
 place_of_supply      | 0   
 delivery_state       | 0   
 invoice_billing_name | 0   
 po_vendor_name       | 0   



In [0]:
#Duplicate Po


purchase_orders_df.groupBy("po_id") \
    .count() \
    .filter("count > 1") \
    .show()

+-----+-----+
|po_id|count|
+-----+-----+
+-----+-----+



In [0]:
#Duplicate Vendor

vendors_df.groupBy("vendor_id") \
    .count() \
    .filter("count > 1") \
    .show()


+---------+-----+
|vendor_id|count|
+---------+-----+
+---------+-----+



In [0]:
#GSTIN Length


from pyspark.sql import functions as F

vendors_df.select(
    F.length("gstin").alias("gstin_length")
).groupBy(
    "gstin_length"
).count().show()

+------------+-----+
|gstin_length|count|
+------------+-----+
|          15|   80|
+------------+-----+



In [0]:
#Date Format check

purchase_orders_df.select(
    "po_date",
    "ewb_generated_date"
).show(5, False)

+----------+------------------+
|po_date   |ewb_generated_date|
+----------+------------------+
|05/01/2023|06/01/2023        |
|05/12/2011|03/02/2024        |
|01/08/2023|NULL              |
|02/03/2023|18/03/2024        |
|23/08/2023|24/08/2023        |
+----------+------------------+
only showing top 5 rows


In [0]:
print("Spark Version:", spark.version)

Spark Version: 4.1.0


In [0]:
print("vendors_df:", vendors_df.count())
print("purchase_orders_df:", purchase_orders_df.count())
print("vendor_invoices_df:", vendor_invoices_df.count())
print("hsn_rate_schedule_df:", hsn_rate_schedule_df.count())
print("cbic_blacklist_df:", cbic_blacklist_df.count())
print("ground_truth_df:", ground_truth_df.count())
print("historical_po_values_df:", historical_po_values_df.count())

vendors_df: 80
purchase_orders_df: 1000
vendor_invoices_df: 1000
hsn_rate_schedule_df: 20
cbic_blacklist_df: 4
ground_truth_df: 1000
historical_po_values_df: 657


##DATA EXPANSION

In [0]:
#raw Expanded

from pyspark.sql import functions as F

target_vendors = 1000

vendor_multiplier = int(target_vendors / vendors_df.count()) + 1

raw_vendors_df = (
    vendors_df
    .crossJoin(
        spark.range(vendor_multiplier)
    )
    .limit(target_vendors)
)

In [0]:
#Add Director Overlap Injection




from pyspark.sql import functions as F

raw_vendors_df = raw_vendors_df.withColumn(
    "director_pan_1",
    F.when(
        F.rand(seed=42) < 0.05,
        F.lit("ABCDE1234F")
    ).otherwise(
        F.col("director_pan_1")
    )
)

In [0]:
#Unique Vendor ID

from pyspark.sql import functions as F
from pyspark.sql.window import Window

w = Window.orderBy(F.monotonically_increasing_id())

raw_vendors_df = raw_vendors_df.withColumn(
    "vendor_id",
    F.concat(
        F.lit("V"),
        F.lpad(F.row_number().over(w), 5, "0")
    )
)


raw_vendors_df.select("vendor_id").distinct().count()

1000

In [0]:
#Unique GSTIN


raw_vendors_df = raw_vendors_df.withColumn(
    "gstin",
    F.concat(
        F.substring("gstin", 1, 10),
        F.lpad(F.row_number().over(w), 4, "0"),
        F.substring("gstin", 15, 1)
    )
)
raw_vendors_df.select("gstin").distinct().count()

1000

In [0]:
#Create Purchase orders



from pyspark.sql import functions as F

target_pos = 50000

po_multiplier = int(target_pos / purchase_orders_df.count()) + 1

raw_purchase_orders_df = (
    purchase_orders_df
    .crossJoin(
        spark.range(po_multiplier)
    )
    .limit(target_pos)
)


raw_purchase_orders_df.count()

50000

In [0]:
cancelled_vendors = (
    raw_vendors_df
    .orderBy(F.rand(seed=42))
    .limit(50)
    .select("vendor_id")
)

raw_purchase_orders_df = (
    raw_purchase_orders_df
    .join(
        cancelled_vendors.withColumn(
            "status",
            F.lit("Cancelled")
        ),
        "vendor_id",
        "left"
    )
)

In [0]:
#Make Po id unique


from pyspark.sql.window import Window
from pyspark.sql import functions as F

w_po = Window.orderBy(F.monotonically_increasing_id())

raw_purchase_orders_df = raw_purchase_orders_df.withColumn(
    "po_id",
    F.concat(
        F.lit("PO"),
        F.lpad(
            F.row_number().over(w_po),
            7,
            "0"
        )
    )
)


raw_purchase_orders_df.select("po_id").distinct().count()

50000

In [0]:
# Vendor Lookup Table
vendor_lookup = (
    raw_vendors_df
    .select("vendor_id")
    .withColumn(
        "rn",
        F.row_number().over(
            Window.orderBy("vendor_id")
        )
    )
)

In [0]:
# Random Vendor Assignment
raw_purchase_orders_df = (
    raw_purchase_orders_df
    .drop("vendor_id")   # old vendor_id remove
    .withColumn(
        "rn",
        (F.floor(F.rand(seed=42) * 1000) + 1).cast("int")
    )
)

In [0]:
#Mismatch State


from pyspark.sql import functions as F

raw_purchase_orders_df = raw_purchase_orders_df.withColumn(
    "state_mismatch_seed",
    F.rand(seed=42)
)

raw_purchase_orders_df = raw_purchase_orders_df.withColumn(
    "buyer_state",
    F.when(
        F.col("state_mismatch_seed") < 0.006,  # ~300 records out of 50000
        F.lit("Karnataka")
    ).otherwise(
        F.col("buyer_state")
    )
).drop("state_mismatch_seed")

In [0]:
###


raw_purchase_orders_df = raw_purchase_orders_df.withColumn(
    "rn",
    (
        (F.rand(seed=42) * 1000).cast("int") + 1
    )
)

# Join
raw_purchase_orders_df = (
    raw_purchase_orders_df
    .join(
        vendor_lookup,
        on="rn",
        how="left"
    )
    .drop("rn")
)

In [0]:
print("PO Rows =", raw_purchase_orders_df.count())

print(
    "Unique PO IDs =",
    raw_purchase_orders_df.select("po_id").distinct().count()
)

print(
    "Vendor IDs Used =",
    raw_purchase_orders_df.select("vendor_id").distinct().count()
)

PO Rows = 50000
Unique PO IDs = 50000
Vendor IDs Used = 1000


In [0]:
#Create Vendor invoices



from pyspark.sql import functions as F

target_invoices = 50000

invoice_multiplier = int(
    target_invoices / vendor_invoices_df.count()
) + 1

raw_vendor_invoices_df = (
    vendor_invoices_df
    .crossJoin(
        spark.range(invoice_multiplier)
    )
    .limit(target_invoices)
)



raw_vendor_invoices_df.count()

50000

In [0]:
#Make Invoice Numbers Unique




from pyspark.sql.window import Window
from pyspark.sql import functions as F

w_inv = Window.orderBy(
    F.monotonically_increasing_id()
)

raw_vendor_invoices_df = raw_vendor_invoices_df.withColumn(
    "invoice_number",
    F.concat(
        F.lit("INV"),
        F.lpad(
            F.row_number().over(w_inv),
            8,
            "0"
        )
    )
)#

In [0]:
#MAKE invoices to poids


po_lookup = (
    raw_purchase_orders_df
    .select("po_id")
    .withColumn(
        "rn",
        F.row_number().over(
            Window.orderBy("po_id")
        )
    )
)

raw_vendor_invoices_df = raw_vendor_invoices_df.withColumn(
    "rn",
    F.row_number().over(
        Window.orderBy(
            F.monotonically_increasing_id()
        )
    )
)

raw_vendor_invoices_df = (
    raw_vendor_invoices_df
    .drop("po_id")
    .join(po_lookup, "rn")
    .drop("rn")
)

In [0]:
print(
    "Invoice Rows:",
    raw_vendor_invoices_df.count()
)

print(
    "Unique Invoice Numbers:",
    raw_vendor_invoices_df.select(
        "invoice_number"
    ).distinct().count()
)

print(
    "Unique PO IDs:",
    raw_vendor_invoices_df.select(
        "po_id"
    ).distinct().count()
)

Invoice Rows: 50000
Unique Invoice Numbers: 50000
Unique PO IDs: 50000


##Historical

In [0]:
#Create Historical PO value



target_history = 12000

history_multiplier = int(
    target_history /
    historical_po_values_df.count()
) + 1

raw_historical_po_values_df = (
    historical_po_values_df
    .crossJoin(
        spark.range(history_multiplier)
    )
    .limit(target_history)
)

raw_historical_po_values_df.count()

12000

In [0]:
print("Historical Count:", raw_historical_po_values_df.count())

Historical Count: 12000


In [0]:
raw_historical_po_values_df.groupBy(
    "vendor_id",
    "year_month"
).count().filter(
    F.col("count") > 1
).count()

657

In [0]:
#Assign to vendor ID


vendor_lookup = (
    raw_vendors_df
    .select("vendor_id")
    .withColumn(
        "rn",
        F.row_number().over(
            Window.orderBy("vendor_id")
        )
    )
)

In [0]:
raw_historical_po_values_df = (
    raw_historical_po_values_df
    .drop("vendor_id")
)

raw_historical_po_values_df = raw_historical_po_values_df.withColumn(
    "rn",
    (
        (F.rand(seed=42) * 1000)
        .cast("int") + 1
    )
)

raw_historical_po_values_df = (
    raw_historical_po_values_df
    .join(vendor_lookup, "rn", "left")
    .drop("rn")
)

##Anomaly Inject

In [0]:
from pyspark.sql.functions import (
    col,
    lit,
    rand,
    when,
    concat,
    lpad,
    current_date,
    date_sub
)

In [0]:
from pyspark.sql import functions as F

bool_cols = [
    "composition_flag",
    "filing_history_q1",
    "filing_history_q2",
    "filing_history_q3",
    "filing_history_q4",
    "filing_history_q5",
    "filing_history_q6"
]

for c in bool_cols:
    raw_vendors_df = raw_vendors_df.withColumn(
        c,
        F.when(F.col(c).isin("✓", "✔", "Y", "Yes", "TRUE", "true"), 1)
         .when(F.col(c).isin("✗", "✘", "N", "No", "FALSE", "false"), 0)
         .otherwise(F.lit(0))
    )

In [0]:
#GST-005 Composition Dealer (~80)

raw_vendors_df = raw_vendors_df.withColumn(
    "composition_flag",
    when(rand(200) < 0.08, 1)
    .otherwise(col("composition_flag"))
)

In [0]:
#ITC-001 Non Filer (~400)

for q in range(1,7):
    raw_vendors_df = raw_vendors_df.withColumn(
        f"filing_history_q{q}",
        when(rand(q*10) < 0.4, 0)
        .otherwise(col(f"filing_history_q{q}"))
    )

In [0]:
from pyspark.sql.functions import to_date, col

raw_vendors_df = raw_vendors_df.withColumn(
    "registration_date",
    to_date(col("registration_date"), "dd/MM/yyyy")
)

In [0]:
#FRD-005 Fly By Night Vendor (~60)

raw_vendors_df = raw_vendors_df.withColumn(
    "registration_date",
    when(
        rand(1000) < 0.06,
        date_sub(current_date(), 30)
    ).otherwise(col("registration_date"))
)

In [0]:
#TAX-001 HSN Rate Mismatch (~350)

raw_purchase_orders_df = raw_purchase_orders_df.withColumn(
    "cgst_rate",
    when(
        rand(400) < 0.007,
        col("cgst_rate") + 5
    ).otherwise(col("cgst_rate"))
)

In [0]:
#TAX-002 Wrong Tax Type (~200)

raw_purchase_orders_df = raw_purchase_orders_df.withColumn(
    "igst_rate",
    when(
        rand(500) < 0.004,
        18
    ).otherwise(col("igst_rate"))
)

In [0]:
#EWB-001 Missing EWB (~500)

raw_purchase_orders_df = raw_purchase_orders_df.withColumn(
    "ewb_number",
    when(
        rand(600) < 0.01,
        None
    ).otherwise(col("ewb_number"))
)

In [0]:
#EWB-002 Quantity Mismatch (~180)


raw_purchase_orders_df = raw_purchase_orders_df.withColumn(
    "quantity",
    when(
        rand(700) < 0.0036,
        col("quantity") * 10
    ).otherwise(col("quantity"))
)

In [0]:
#FRD-002 Circular Billing


from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

cluster_vendors = (
    raw_vendors_df
    .orderBy(rand(42))
    .limit(120)   # 30 clusters × 4 vendors
    .withColumn(
        "cluster_id",
        ((row_number().over(Window.orderBy("vendor_id")) - 1) / 4).cast("int")
    )
)

shared_pans = (
    cluster_vendors
    .select("cluster_id")
    .distinct()
    .withColumn(
        "shared_pan_1",
        concat(
            lit("AAAAA"),
            lpad(col("cluster_id"), 6, "0")
        )
    )
    .withColumn(
        "shared_pan_2",
        concat(
            lit("BBBBB"),
            lpad(col("cluster_id"), 6, "0")
        )
    )
)

cluster_vendors = (
    cluster_vendors
    .join(shared_pans, "cluster_id")
    .drop("director_pan_1", "director_pan_2")
    .withColumnRenamed("shared_pan_1", "director_pan_1")
    .withColumnRenamed("shared_pan_2", "director_pan_2")
)

remaining_vendors = raw_vendors_df.join(
    cluster_vendors.select("vendor_id"),
    "vendor_id",
    "left_anti"
)

raw_vendors_df = remaining_vendors.unionByName(
    cluster_vendors.select(raw_vendors_df.columns)
)

In [0]:
#FRD-003 Split Invoice


split_po = (
    raw_purchase_orders_df
    .orderBy(rand(99))
    .limit(240)
)

split_po = (
    split_po
    .withColumn("base_amount", lit(30000.0))
    .withColumn("cgst_amt", lit(2700.0))
    .withColumn("sgst_amt", lit(2700.0))
    .withColumn("igst_amt", lit(0.0))
    .withColumn("total_amount", lit(35400.0))
)

split_po_1 = (
    split_po
    .withColumn(
        "po_id",
        concat(col("po_id"), lit("_A"))
    )
)

split_po_2 = (
    split_po
    .withColumn(
        "po_id",
        concat(col("po_id"), lit("_B"))
    )
)

raw_purchase_orders_df = (
    raw_purchase_orders_df
    .unionByName(split_po_1)
    .unionByName(split_po_2)
)


print("FRD-003 Split Invoice Added")

FRD-003 Split Invoice Added


In [0]:
#FRD-004 Value Spike (~90)



raw_purchase_orders_df = raw_purchase_orders_df.withColumn(
    "base_amount",
    when(
        rand(800) < 0.0018,
        col("base_amount") * 4
    ).otherwise(col("base_amount"))
)

In [0]:
#SIM-001 Name Mismatch (~600)


raw_purchase_orders_df = raw_purchase_orders_df.withColumn(
    "invoice_billing_name",
    when(
        rand(900) < 0.012,
        lit("XYZ TRADING COMPANY")
    ).otherwise(col("invoice_billing_name"))
)

In [0]:
#ITC-002 Excess ITC Claim (~250)

raw_vendor_invoices_df = raw_vendor_invoices_df.withColumn(
    "itc_claimed_by_buyer",
    when(
        rand(300) < 0.005,
        col("gstr2b_itc_available") * 1.5
    ).otherwise(col("itc_claimed_by_buyer"))
)

In [0]:
#DQ-001 Duplicate Invoice (~100)



duplicate_rows = raw_vendor_invoices_df.limit(100)

raw_vendor_invoices_df = raw_vendor_invoices_df.union(
    duplicate_rows
)

In [0]:
from pyspark.sql import functions as F

flag_cols = [
    "composition_flag",
    "filing_history_q1",
    "filing_history_q2",
    "filing_history_q3",
    "filing_history_q4",
    "filing_history_q5",
    "filing_history_q6"
]

for c in flag_cols:
    raw_vendors_df = raw_vendors_df.withColumn(
        c,
        F.col(c).cast("double").cast("int")
    )

raw_historical_po_values_df = (
    raw_historical_po_values_df
    .withColumn(
        "po_count",
        F.col("po_count").cast("double").cast("int")
    )
)

##Ground truth

In [0]:
from pyspark.sql import functions as F

raw_ground_truth_df = (
    raw_purchase_orders_df
    .select("po_id")
    .withColumn("r", F.rand(42))
    .withColumn(
        "anomaly_code",
        F.when(F.col("r") < 0.006, "GST-001")
        .when(F.col("r") < 0.012, "GST-003")
        .when(F.col("r") < 0.015, "GST-004")
        .when(F.col("r") < 0.017, "GST-005")
        .when(F.col("r") < 0.025, "ITC-001")
        .when(F.col("r") < 0.030, "ITC-002")
        .when(F.col("r") < 0.037, "TAX-001")
        .when(F.col("r") < 0.041, "TAX-002")
        .when(F.col("r") < 0.051, "EWB-001")
        .when(F.col("r") < 0.055, "EWB-002")
        .when(F.col("r") < 0.057, "FRD-002")
        .when(F.col("r") < 0.0585, "FRD-003")
        .when(F.col("r") < 0.060, "FRD-004")
        .otherwise("CLEAN")
    )
    .withColumn(
        "is_anomalous",
        F.when(F.col("anomaly_code") == "CLEAN", 0).otherwise(1)
    )
    .withColumn(
        "severity",
        F.when(F.col("anomaly_code") == "CLEAN", "Low")
        .otherwise("High")
    )
    .drop("r")
)

In [0]:
print(
    "Ground Truth Rows:",
    raw_ground_truth_df.count()
)

print(
    "Unique PO IDs:",
    raw_ground_truth_df.select(
        "po_id"
    ).distinct().count()
)

Ground Truth Rows: 50480
Unique PO IDs: 50480


In [0]:
raw_ground_truth_df.groupBy(
    "is_anomalous"
).count().show()

+------------+-----+
|is_anomalous|count|
+------------+-----+
|           1| 3073|
|           0|47407|
+------------+-----+



In [0]:
raw_ground_truth_df.groupBy(
    "anomaly_code"
).count().orderBy(
    "count",
    ascending=False
).show(50,False)

+------------+-----+
|anomaly_code|count|
+------------+-----+
|CLEAN       |47407|
|EWB-001     |495  |
|ITC-001     |396  |
|TAX-001     |378  |
|GST-001     |330  |
|GST-003     |298  |
|ITC-002     |245  |
|TAX-002     |207  |
|EWB-002     |194  |
|GST-004     |156  |
|FRD-002     |118  |
|GST-005     |114  |
|FRD-004     |79   |
|FRD-003     |63   |
+------------+-----+



##Cbic Blacklist

In [0]:
from pyspark.sql import functions as F

cbic_blacklist_df = (
    raw_vendors_df
    .orderBy(F.rand(42))
    .limit(50)
    .select("gstin")
    .withColumn("blacklist_date", F.current_date())
    .withColumn("reason", F.lit("Fake GST Registration"))
    .withColumn("case_reference", F.lit("CBIC-TEST"))
)

##Verify


In [0]:
raw_ground_truth_df.groupBy(
    "is_anomalous"
).count().show()

+------------+-----+
|is_anomalous|count|
+------------+-----+
|           1| 3073|
|           0|47407|
+------------+-----+



In [0]:
print("raw_vendors_df =", raw_vendors_df.count())
print("raw_purchase_orders_df =", raw_purchase_orders_df.count())
print("raw_vendor_invoices_df =", raw_vendor_invoices_df.count())
print("raw_ground_truth_df =", raw_ground_truth_df.count())
print("raw_historical_po_values_df =", raw_historical_po_values_df.count())
print("hsn_rate_schedule_df =", hsn_rate_schedule_df.count())
print("cbic_blacklist_df =", cbic_blacklist_df.count())

raw_vendors_df = 1000
raw_purchase_orders_df = 50480
raw_vendor_invoices_df = 50100
raw_ground_truth_df = 50480
raw_historical_po_values_df = 12000
hsn_rate_schedule_df = 20
cbic_blacklist_df = 50


In [0]:
raw_historical_po_values_df.select("vendor_id").distinct().count()

1000

##RAW DATA SAVE


In [0]:
raw_vendors_df.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/gst-fraud&anomaly_detection/raw/vendors"
)

raw_purchase_orders_df.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/gst-fraud&anomaly_detection/raw/purchase_orders"
)

raw_vendor_invoices_df.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/gst-fraud&anomaly_detection/raw/vendor_invoices"
)

raw_ground_truth_df.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/gst-fraud&anomaly_detection/raw/ground_truth"
)

raw_historical_po_values_df.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/gst-fraud&anomaly_detection/raw/historical_po_values"
)

hsn_rate_schedule_df.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/gst-fraud&anomaly_detection/raw/hsn_rate_schedule"
)

cbic_blacklist_df.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/gst-fraud&anomaly_detection/raw/cbic_blacklist"
)

In [0]:
display(
    dbutils.fs.ls(
        "/Volumes/workspace/default/gst-fraud&anomaly_detection/raw/"
    )
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/gst-fraud&anomaly_detection/raw/cbic_blacklist/,cbic_blacklist/,0,1781687468815
dbfs:/Volumes/workspace/default/gst-fraud&anomaly_detection/raw/ground_truth/,ground_truth/,0,1781687468815
dbfs:/Volumes/workspace/default/gst-fraud&anomaly_detection/raw/historical_po_values/,historical_po_values/,0,1781687468815
dbfs:/Volumes/workspace/default/gst-fraud&anomaly_detection/raw/hsn_rate_schedule/,hsn_rate_schedule/,0,1781687468816
dbfs:/Volumes/workspace/default/gst-fraud&anomaly_detection/raw/purchase_orders/,purchase_orders/,0,1781687468816
dbfs:/Volumes/workspace/default/gst-fraud&anomaly_detection/raw/testground_truth/,testground_truth/,0,1781687468816
dbfs:/Volumes/workspace/default/gst-fraud&anomaly_detection/raw/testhistorical_po_values/,testhistorical_po_values/,0,1781687468816
dbfs:/Volumes/workspace/default/gst-fraud&anomaly_detection/raw/testvendor_invoices/,testvendor_invoices/,0,1781687468816
dbfs:/Volumes/workspace/default/gst-fraud&anomaly_detection/raw/testvendors/,testvendors/,0,1781687468816
dbfs:/Volumes/workspace/default/gst-fraud&anomaly_detection/raw/vendor_invoices/,vendor_invoices/,0,1781687468816


In [0]:
# vendors_df.write.mode("overwrite").parquet(
#     "/Volumes/workspace/default/gst-fraud&anomaly_detection/raw/vendors"
# )
# purchase_orders_df.write.mode("overwrite").parquet(
#     "/Volumes/workspace/default/gst-fraud&anomaly_detection/raw/purchase_orders"
# )
# vendor_invoices_df.write.mode("overwrite").parquet(
#     "/Volumes/workspace/default/gst-fraud&anomaly_detection/raw/vendor_invoices"
# )
# hsn_rate_schedule_df.write.mode("overwrite").parquet(
#     "/Volumes/workspace/default/gst-fraud&anomaly_detection/raw/hsn_rate_schedule"
# )
# cbic_blacklist_df.write.mode("overwrite").parquet(
#     "/Volumes/workspace/default/gst-fraud&anomaly_detection/raw/cbic_blacklist"
# )
# ground_truth_df.write.mode("overwrite").parquet(
#     "/Volumes/workspace/default/gst-fraud&anomaly_detection/raw/ground_truth"
# )
# historical_po_values_df.write.mode("overwrite").parquet(
#     "/Volumes/workspace/default/gst-fraud&anomaly_detection/raw/historical_po_values"
# )



# display(
#     dbutils.fs.ls(
#         "/Volumes/workspace/default/gst-fraud&anomaly_detection/raw/"
#     )
# )